# VN30F1M — backtest results (5 phút)

Notebook chạy CANSLIM qua API, rồi trình bày input → summary → fills/trades → vị thế → audit → equity của cùng run_id. Agent không phải dependency.

Khởi động API bằng `python -m uvicorn backtest_hpg.intraday_main:app --host 127.0.0.1 --port 8770` sau khi import snapshot và chuẩn bị policy tĩnh. Xem [hướng dẫn](../docs/plans/vn30f1m-backtest-runbook.md) và [checklist local](../.agents/checklists/vn30f1m-backtest-checklist.md).

**Normalized simulation:** giá VN30F1M giữ index points nguồn; quantity là đơn vị mô phỏng, không mặc nhiên là số hợp đồng/P&L futures thực tế. Timestamp Open; Close/available_at = Open + 5 phút theo assumption user. Giữ record ATC 14:45; cửa sổ 200/65/50 nến liên tục qua phiên; không fallback daily. Vị thế giữ qua đáo hạn theo map tham khảo đã được user cho phép.

Payload pin bundle thật bên dưới. API fail rõ nếu kỳ báo cáo thiếu expected session/bar hoặc thiếu map; không tự chuyển range để làm run pass. Evaluation status phân biệt thiếu warm-up và no-trade.


In [ ]:
import json
import os
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
API_URL = os.getenv("BACKTEST_API_URL", "http://127.0.0.1:8770").rstrip("/")
DATASET_MANIFEST = Path(os.getenv(
    "BACKTEST_DATASET_MANIFEST",
    str(PROJECT_ROOT / "data/backtest-store/datasets/dataset-52ca9fbe68ce00878bf9cc10d65a088e47eb80b84086f4ec26deb67a1f4e4f9b.json"),
))
dataset_manifest = json.loads(DATASET_MANIFEST.read_text(encoding="utf-8"))
payload = {
    "dataset_id": dataset_manifest["dataset_id"],
    "dataset_version": dataset_manifest["dataset_version"],
    "symbol": "VN30F1M",
    "start_date": os.getenv("BACKTEST_REPORT_START", "2026-03-15"),
    "end_date": os.getenv("BACKTEST_REPORT_END", "2026-09-15"),
    "strategy_id": "canslim_breakout_v0",
    "initial_cash": "10000000",
    "fee_rate": "0.001",
    "slippage_rate": "0.002",
}


In [ ]:
request = Request(
    f"{API_URL}/api/backtests",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
try:
    with urlopen(request, timeout=120) as response:
        result = json.load(response)
except HTTPError as error:
    raise RuntimeError(f"API {error.code}: {error.read().decode('utf-8')}") from error

required_groups = {"metadata", "summary", "fills", "trades", "open_position", "signals", "orders", "equity_history", "evaluations", "evaluation_status"}
missing_groups = required_groups - result.keys()
if missing_groups:
    raise RuntimeError(f"Response thiếu nhóm dữ liệu: {sorted(missing_groups)}")
with urlopen(f"{API_URL}/api/backtests/{result['metadata']['run_id']}", timeout=120) as response:
    reloaded_result = json.load(response)
if reloaded_result != result:
    raise RuntimeError("GET result khác POST result của cùng run_id")


## 1. Run dataset

- `run_id`: định danh duy nhất của lần chạy.
- `dataset_id`, `dataset_version`, `content_hash`: xác định chính xác snapshot đầu vào.
- `engine_version`: phiên bản bộ máy backtest tạo kết quả.
- **Run config**: symbol, kỳ báo cáo, vốn, phí và slippage đã sử dụng.
- **Strategy parameters**: các threshold/rule cố định của strategy.
- `label` (`"normalized simulation"`): kết quả dùng mô hình tiền normalized; P/L không mặc nhiên đại diện số tiền giao dịch lịch sử thực tế.


In [ ]:
metadata = result["metadata"]
dataset_df = pd.DataFrame([{
    key: metadata[key]
    for key in ("run_id", "label", "dataset_id", "dataset_version", "content_hash", "engine_version", "source_timeframe", "strategy_timeframe", "execution_timeframe", "timezone", "policy_hash")
}])
display(dataset_df)
display(Markdown("**Input snapshots**"), pd.DataFrame(metadata["datasets"])[
    ["symbol", "timeframe", "dataset_version", "content_hash", "source_url", "count", "start", "end", "extracted_at", "imported_at"]
])
display(Markdown("**Run config**"), pd.DataFrame([metadata["config"]]))
display(Markdown("**Strategy parameters — window unit: 5-minute bars**"), pd.DataFrame([metadata["strategy_parameters"]]))
display(Markdown("**Warm-up trước kỳ báo cáo**"), pd.DataFrame([metadata["warmup_bars"]]))
display(Markdown("**Warm-up range**"), pd.DataFrame(metadata["warmup_range"]).T.reset_index(names="symbol"))
display(Markdown("**Assumptions**"), pd.DataFrame([metadata["assumptions"]]))
display(Markdown("**Evaluation coverage**"), pd.DataFrame([result["evaluation_status"]]))


## 2. Kết quả tổng quan

Các chỉ số trả lời trực tiếp strategy làm thay đổi tài khoản thế nào:

- `initial_cash`: vốn ban đầu.
- `final_equity`: tiền mặt + giá trị thị trường cuối kỳ.
- `realized_pnl`: lãi/lỗ đã chốt từ các vị thế đóng.
- `unrealized_pnl`: lãi/lỗ chưa chốt của vị thế còn mở.
- `total_return`: `(final_equity − initial_cash) / initial_cash`; dương là tăng, âm là giảm.


In [ ]:
summary = result["summary"]
summary_view = {**summary, "total_return": f"{float(summary['total_return']):.2%}"}
display(pd.DataFrame([summary_view]))
display(Markdown(
    "### Kết luận nhanh\n"
    f"- **Final equity:** `{summary['final_equity']}` từ vốn ban đầu `{summary['initial_cash']}`.\n"
    f"- **Total return:** `{summary_view['total_return']}`.\n"
    f"- **P/L:** realized `{summary['realized_pnl']}`, unrealized `{summary['unrealized_pnl']}`.\n"
    f"- **Hoạt động:** `{len(result['fills'])}` fills tạo `{len(result['trades'])}` closed trades.\n"
    f"- **Cuối kỳ:** {'còn open position' if result['open_position'] else 'không còn open position'}."
))

In [ ]:
def show_table(rows, columns):
    frame = pd.DataFrame(rows, columns=columns)
    display(frame if not frame.empty else Markdown("_Không có dữ liệu._"))

## 3. Lệnh đã khớp (fills)

Mỗi dòng là một giao dịch thực sự làm thay đổi cash hoặc position:

- `fill_time`: timestamp Open nến hợp lệ kế tiếp sau signal, có UTC offset.
- `side`: BUY mở vị thế, SELL đóng vị thế.
- `fill_price`: giá khớp mô phỏng đã phản ánh slippage.
- `quantity`: số lượng đơn vị mô phỏng khớp.
- `fee`: phí một chiều của lần khớp này (chưa gộp với chiều kia).

> Signal chưa khớp không xuất hiện trong bảng này.


In [ ]:
show_table(result["fills"], ["fill_time", "side", "fill_price", "quantity", "fee"])

## 4. Giao dịch đã đóng (closed trades)

Mỗi dòng ghép một BUY fill với SELL fill thành một vòng giao dịch hoàn chỉnh:

- `entry_date`, `exit_date`: timestamp khớp mua và bán, không phải ngày signal.
- `quantity`: số đơn vị mô phỏng của giao dịch.
- `fees`: tổng phí hai chiều (BUY + SELL) của giao dịch.
- `net_pnl`: lãi/lỗ ròng sau khi trừ `fees`.
- `close_reason`: lý do SELL signal phát sinh (`STOP_LOSS` hoặc `TAKE_PROFIT`).

> Bảng rỗng vẫn có thể là run hợp lệ: strategy không giao dịch hoặc vị thế chưa đóng.


In [ ]:
show_table(result["trades"], ["entry_date", "exit_date", "quantity", "net_pnl", "close_reason"])

## 5. Vị thế cuối kỳ

Bảng này cho biết cuối kỳ tài khoản còn giữ vị thế hay không:

- `quantity`, `entry_price`: quy mô và giá khớp mua của vị thế còn mở.
- `entry_pivot`: pivot tại thời điểm mua — mốc tham chiếu tính TAKE_PROFIT.
- `stop_reference`: giá tham chiếu stop loss (`entry_price × (1 − STOP_LOSS_PCT)`).
- `market_value`: giá trị vị thế theo Close cuối cùng.
- `unrealized_pnl`: lãi/lỗ chưa chốt tính từ cost basis (entry_price × quantity + entry_fee), không cộng vào realized P/L.
- Không có dữ liệu: tài khoản đã về trạng thái không nắm giữ.

> Engine không tạo SELL giả để đóng vị thế vào ngày cuối dataset.


In [ ]:
position = [result["open_position"]] if result["open_position"] else []
show_table(position, ["quantity", "entry_price", "market_value", "unrealized_pnl"] )

## 6. Audit signal và order

Phần audit giải thích đường đi từ quyết định strategy tới execution:

- **Signal** được tạo sau Close; `reason` và `pivot` ghi lại lý do quyết định.
- **Order** được tạo từ signal và chờ xử lý ở nến hợp lệ kế tiếp.
- `filled`: order đã tạo fill.
- `rejected`: order bị từ chối; xem `rejection_reason`.
- `unfilled`: signal phát sinh ở nến cuối, không còn nến hợp lệ kế tiếp để khớp.

> Chuỗi đúng là **signal sau Close → order → fill ở Open kế tiếp**; signal không đồng nghĩa giao dịch đã khớp.


In [ ]:
display(Markdown("**Signals**"))
show_table(result["signals"], ["signal_time", "side", "reason", "pivot"])
display(Markdown("**Orders**"))
show_table(result["orders"], ["created_time", "side", "status", "rejection_reason"])
display(Markdown("**Strategy evaluation — UNEVALUABLE không phải rule đã fail/pass**"))
show_table(result["evaluations"], ["time", "status", "reason", "market_sample_count", "market_available_at", "market_close", "indicators"])


## 7. Diễn biến tài khoản (equity history)

Mỗi dòng là snapshot tài khoản tại Close của một nến 5 phút:

- `cash`: tiền mặt tại Close — fill xảy ra ở Open cùng nến đó đã được phản ánh.
- `quantity`: số đơn vị mô phỏng đang giữ.
- `market_value`: giá trị vị thế theo Close.
- `equity`: `cash + market_value`.
- `unrealized_pnl`: lãi/lỗ chưa chốt của vị thế hiện tại.


In [ ]:
show_table(result["equity_history"], ["trading_date", "cash", "quantity", "market_value", "equity", "unrealized_pnl"])